# Stage-1 Interpreter — FULL run (all 2761 LUTs) + INTENSITY-fix test → Stage-13A eval

Fresh **A100 + High-RAM**, set tokens in CELL 1, Run All (or run cells as needed).

- **CELL 1–4:** the full 2761-LUT run (caption → corpus → train `interp_full` → score). ~2.5–4 h.
- **CELL 5–8:** the intensity-fix test — re-caption a 500-LUT slice with INTENSITY-aware prompts
  (the full run found magnitude unlearnable because captions were intensity-free), retrain
  `interp_intensity`, **persist to HF after training**, and score. Reuses the 500/500 supplement.

Everything is resumable. THE number for the intensity test is `attribute_bucket_f1[real_lut]`
(CELL 8) vs the full run's 0.16: a jump to ~0.4+ means intensity-aware captions fixed grade
magnitude; staying ~0.16 means fall back to a router-only interpreter.

In [ ]:
# CELL 1 - provision (clone, install, tokens, stage corpus for active_rows.jsonl)
import os, pathlib, subprocess, sys
REPO, BRANCH = '/content/SLM', 'feat/two-stage'
if not pathlib.Path(REPO, '.git').is_dir():
    subprocess.run(['git', 'clone', 'https://github.com/ericrcwu001/SLM', REPO], check=True)
os.chdir(REPO)
subprocess.run(['git', 'fetch', 'origin', BRANCH], check=True)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['git', 'reset', '--hard', f'origin/{BRANCH}'], check=True)
if not os.environ.get('SLM_PROVISIONED'):
    # [frontier] brings the openai SDK the teacher gateway needs; [sft] covers torch/transformers.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[sft,frontier]'], check=True)
    os.environ['SLM_PROVISIONED'] = '1'
print('HEAD:', subprocess.run(['git','log','--oneline','-1'], capture_output=True, text=True).stdout.strip())

def _env_token(name):
    for _p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(_p)
        if fp.is_file():
            for _l in fp.read_text().splitlines():
                if _l.strip().startswith(name + '='):
                    return _l.split('=', 1)[1].strip().strip('"').strip("'")
    return None

# ALWAYS re-prompt (Enter=keep the current value; type a new one to fix a wrong entry).
import getpass
_DEFAULTS = {'TFY_BASE_URL': 'https://tfy.promptlens.trilogy.com/v1'}
for _n in ('HF_TOKEN', 'HF_WRITE_TOKEN', 'TFY_BASE_URL', 'TFY_API_KEY'):
    _cur = os.environ.get(_n) or _env_token(_n) or _DEFAULTS.get(_n, '')
    if not _cur:
        _hint = ' [optional, Enter to skip]' if _n == 'HF_WRITE_TOKEN' else ' [required]'
    elif _n == 'TFY_BASE_URL':
        _hint = f' [Enter=keep {_cur}]'
    else:
        _hint = f' [Enter=keep ...{_cur[-4:]}]'
    os.environ[_n] = getpass.getpass(f'{_n}{_hint}: ').strip() or _cur
print('HF_TOKEN set:', bool(os.environ.get('HF_TOKEN')),
      '| HF_WRITE set:', bool(os.environ.get('HF_WRITE_TOKEN')),
      '| TFY ready:', bool(os.environ.get('TFY_BASE_URL')) and bool(os.environ.get('TFY_API_KEY')))

os.environ['SLM_ARTIFACT_ROOT'] = '/content/slm'
if not pathlib.Path('data/active_sft/active_rows.jsonl').is_file():
    subprocess.run(['slm_stage', 'stage', '--durable-root', 'hf://datasets/ericrcwu/LUT_SLM',
                    '--local-root', '/content/slm'], check=True)
print('active_rows.jsonl:', pathlib.Path('data/active_sft/active_rows.jsonl').is_file())

In [ ]:
# CELL 2 - FULL data: caption ALL 2761 LUTs + supplement (500/500) + unified corpus. ~1-2h, RESUMABLE.
import subprocess, sys, json
assert subprocess.run([sys.executable, '-m', 'scripts.generate_captions',
                       '--no-image', '--workers', '6']).returncode == 0, 'caption gen failed / handed off'
assert subprocess.run([sys.executable, '-m', 'scripts.generate_route_supplement']).returncode == 0, 'supplement failed'
assert subprocess.run([sys.executable, '-m', 'scripts.build_interpreter_corpus']).returncode == 0, 'corpus build failed'
print(json.dumps(json.load(open('data/interpreter/interpreter_corpus_manifest.json')), indent=2))

In [ ]:
# CELL 3 - train the interpreter on the FULL corpus (A100 bf16), run-id interp_full. ~30-60 min.
import subprocess, sys, glob
assert subprocess.run([sys.executable, '-m', 'interpreter.train', '--config',
                       'configs/candidate_interpreter.json', '--run-id', 'interp_full']).returncode == 0, 'train failed'
print('trained:', sorted(glob.glob('models/interpreter/interp_full_*'))[-1])

In [ ]:
# CELL 4 - score the FULL model on the untouched holdout (captured so the summary always prints).
# ~15-30 min (full holdout ~800+ rows). Add '--limit', '300' for a fast preview.
import subprocess, sys, glob
ADAPTER = sorted(glob.glob('models/interpreter/interp_full_*'))[-1]
r = subprocess.run([sys.executable, '-m', 'interpreter.score', '--config',
                    'configs/candidate_interpreter.json', '--adapter', ADAPTER],
                   capture_output=True, text=True)
print(r.stdout[-3000:]); print(r.stderr[-800:] if r.returncode else '')

In [ ]:
# CELL 5 (INTENSITY TEST) - pull the intensity-aware captioner, caption a 500-LUT slice into a
# SEPARATE cache (does NOT reuse the non-intensity full captions), build the intensity corpus.
# Reuses the 500/500 route supplement already generated in this runtime. ~15-25 min.
import os, subprocess, sys, json, glob
os.chdir('/content/SLM')
subprocess.run(['git', 'fetch', 'origin', 'feat/two-stage', '-q'])
subprocess.run(['git', 'reset', '--hard', 'origin/feat/two-stage', '-q'])
CAP_CACHE = 'data/active_sft/caption_cache_intensity.jsonl'
CAP_OUT   = 'data/active_sft/caption_rows_intensity.jsonl'
CORPUS    = 'data/interpreter/interpreter_rows_intensity.jsonl'
assert subprocess.run([sys.executable, '-m', 'scripts.generate_captions', '--limit', '500',
    '--no-image', '--workers', '6', '--cache', CAP_CACHE, '--out', CAP_OUT]).returncode == 0, 'caption gen failed'
assert subprocess.run([sys.executable, '-m', 'scripts.build_interpreter_corpus',
    '--caption-rows', CAP_OUT, '--out', CORPUS]).returncode == 0, 'corpus build failed'
print(json.dumps(json.load(open('data/interpreter/interpreter_corpus_manifest.json')), indent=2))

In [ ]:
# CELL 6 (INTENSITY TEST) - train on the intensity corpus (A100 bf16), run-id interp_intensity.
import subprocess, sys, glob
assert subprocess.run([sys.executable, '-m', 'interpreter.train', '--config',
    'configs/candidate_interpreter_intensity.json', '--run-id', 'interp_intensity']).returncode == 0, 'train failed'
print('trained:', sorted(glob.glob('models/interpreter/interp_intensity_*'))[-1])

In [ ]:
# CELL 7 (PERSIST AFTER TRAINING) - push the trained interpreter + caches to Hugging Face so a
# runtime teardown can't lose it. Runs BEFORE score so the expensive artifact is safe first.
# Set the repos to yours; needs a WRITE token (SLM_Alpha_Write), not the read HF_TOKEN.
import os, glob, pathlib, getpass
from huggingface_hub import HfApi
MODEL_REPO   = 'ericrcwu/LUT_SLM_interpreter'   # <-- CHANGE to your model repo
DATASET_REPO = 'ericrcwu/LUT_SLM'               # <-- caches/corpus go here
def _envtok(name):
    for p in ('/content/SLM/.env', '/content/.env', '.env'):
        fp = pathlib.Path(p)
        if fp.is_file():
            for l in fp.read_text().splitlines():
                if l.strip().startswith(name + '='):
                    return l.split('=', 1)[1].strip().strip('"').strip("'")
    return None
WTOK = os.environ.get('HF_WRITE_TOKEN') or _envtok('HF_WRITE_TOKEN') or getpass.getpass('HF WRITE token (SLM_Alpha_Write): ').strip()
api = HfApi(token=WTOK)
ADAPTER = sorted(glob.glob('models/interpreter/interp_intensity_*'))[-1]
api.create_repo(MODEL_REPO, repo_type='model', exist_ok=True, private=True)
api.upload_folder(folder_path=ADAPTER, repo_id=MODEL_REPO, repo_type='model', path_in_repo='interp_intensity')
for f in ('data/active_sft/caption_cache_intensity.jsonl', 'data/active_sft/route_supplement_cache.jsonl',
          'data/interpreter/interpreter_rows_intensity.jsonl', 'data/interpreter/interpreter_corpus_manifest.json'):
    if os.path.exists(f):
        api.upload_file(path_or_fileobj=f, path_in_repo='interpreter/' + os.path.basename(f),
                        repo_id=DATASET_REPO, repo_type='dataset')
print('PERSISTED: model ->', MODEL_REPO, '(interp_intensity) ; caches ->', DATASET_REPO)

In [ ]:
# CELL 8 (INTENSITY TEST) - score. THE number: attribute_bucket_f1[real_lut] vs the full run's 0.16.
# >~0.4 => intensity-aware captions fixed magnitude (grade path viable). ~0.16 => router-only fallback.
import subprocess, sys, glob
ADAPTER = sorted(glob.glob('models/interpreter/interp_intensity_*'))[-1]
r = subprocess.run([sys.executable, '-m', 'interpreter.score', '--config',
    'configs/candidate_interpreter_intensity.json', '--adapter', ADAPTER], capture_output=True, text=True)
print(r.stdout[-3500:]); print(r.stderr[-800:] if r.returncode else '')